In [8]:
import json

file_path = 'TD_W34_testinstans.json'

with open(file_path, 'r') as f:
    data = json.load(f)

# Loop through stations and print ID and Location
for station in data['stations']:
    print(f"{station['id']}: {station['location']}")

0: [63.414746566093925, 10.397386363673064]
1: [63.42126270770024, 10.386516005020496]
2: [63.42078238303412, 10.404757641464101]
3: [63.43051170410531, 10.392764550201036]
4: [63.43483884684911, 10.403396522008734]
5: [63.429289773431165, 10.369248388233245]
6: [63.44208952370503, 10.427939146757126]
7: [63.4358846800558, 10.419401004910469]
8: [63.433509009133566, 10.411412715911865]
9: [63.423087556928024, 10.38932740688324]
10: [63.42889745011402, 10.39928913116455]
11: [63.41541842278556, 10.399565398693085]
12: [63.42646180597998, 10.37818819284439]
13: [63.433085554043984, 10.397391468286514]
14: [63.42629161886643, 10.404109150403997]
15: [63.43592006472795, 10.414788275957108]
16: [63.412525462202666, 10.40068656206131]
17: [63.430470910132236, 10.393931418657303]
18: [63.427610017040934, 10.394511072969863]
19: [63.42792321730192, 10.389504432678223]
20: [63.43853720980968, 10.419105291366577]
21: [63.42907021700727, 10.37181057035923]
22: [63.430403126559675, 10.374769717454

In [14]:
import json

input_file = 'TD_W34_testinstans.json'

with open(input_file, 'r') as f:
    data = json.load(f)

# Get the first station to inspect its structure
if data['stations']:
    first_station = data['stations'][0]
    print("Keys available for each station:")
    for key in first_station.keys():
        print(f"- {key}")
else:
    print("No stations found in the file.")

Keys available for each station:
- id
- location
- is_depot
- capacity
- num_bikes
- leave_intensities
- leave_intensities_stdev
- arrive_intensities
- arrive_intensities_stdev
- move_probabilities


In [9]:
import json
import folium

# Load the data
file_path = 'TD_W34_testinstans.json'
with open(file_path, 'r') as f:
    data = json.load(f)

# Center map on the first station
start_coords = data['stations'][0]['location']
m = folium.Map(location=start_coords, zoom_start=14)

# Add markers
for station in data['stations']:
    folium.Marker(
        location=station['location'],
        popup=f"ID: {station['id']}",
        tooltip=f"Station {station['id']}"
    ).add_to(m)

# Display the map directly in the notebook
m

In [18]:
import json

# --- Configuration ---
input_file = 'TD_W34_testinstans.json'
output_file = 'TD_W34_filtered_28_stations.json'

# 1. Define Stations to Keep
# Range 0 to 38 covers IDs 0-37. 
all_ids = list(range(0, 38)) 
stations_to_exclude = [3, 5, 6, 7, 12, 16, 20, 21, 22, 36]
ids_to_include = [s for s in all_ids if s not in stations_to_exclude]

# 2. Define Global Matrices (Top-level keys)
# These are lists of lists that need both rows and columns filtered
global_matrix_keys = [
    'traveltime', 
    'traveltime_stdev', 
    'traveltime_vehicle', 
    'traveltime_vehcile', 
    'bike_travel_time',
    'distance_matrix'
]

# 3. Define Station-Level Lists (Keys inside 'stations')
# These are lists inside a station that point to other stations (e.g., probabilities)
station_dependent_keys = ['move_probabilities']

# --- Processing ---
with open(input_file, 'r') as f:
    data = json.load(f)

# Step A: Identify indices to keep
# We map the old index (0 to 37) to whether we keep it or not
indices_to_keep = []
filtered_stations = []

# We assume the input 'stations' list is sorted by ID 0..37. 
for i, station in enumerate(data['stations']):
    if station['id'] in ids_to_include:
        indices_to_keep.append(i)
        
        # Create a copy of the station to modify
        new_station = station.copy()
        
        # --- CRITICAL FIX: Renumber ID to match new list index ---
        # The simulation expects ID 0 to be at index 0, ID 1 at index 1, etc.
        new_station['id'] = len(filtered_stations) 
        
        # Step B: Filter lists inside the station (like move_probabilities)
        for key in station_dependent_keys:
            if key in new_station and isinstance(new_station[key], list):
                # Only filter if the list length matches the ORIGINAL station count
                # (This prevents double-filtering or filtering irrelevant lists)
                if len(new_station[key]) == len(data['stations']):
                    new_station[key] = [new_station[key][j] for j in indices_to_keep]
                
        filtered_stations.append(new_station)

# Step C: Replace stations list
data['stations'] = filtered_stations

# Step D: Filter Global Matrices
for key in global_matrix_keys:
    if key in data and isinstance(data[key], list):
        print(f"Filtering global matrix: {key}")
        
        # 1. Filter Rows
        filtered_rows = [data[key][i] for i in indices_to_keep]
        
        # 2. Filter Columns (if it's a matrix)
        # Check if the first element is a list
        if len(filtered_rows) > 0 and isinstance(filtered_rows[0], list):
            data[key] = [[row[j] for j in indices_to_keep] for row in filtered_rows]
        else:
            # It was just a list of values (one per station), so rows are enough
            data[key] = filtered_rows

# --- Save ---
with open(output_file, 'w') as f:
    json.dump(data, f, indent=4)

print(f"Success! Created {output_file}")
print(f"Original station count: {len(all_ids)}")
print(f"New station count: {len(filtered_stations)}")
print("Station IDs have been renumbered to 0..27 to match list indices.")

Filtering global matrix: traveltime
Filtering global matrix: traveltime_stdev
Filtering global matrix: traveltime_vehicle
Success! Created TD_W34_filtered_28_stations.json
Original station count: 38
New station count: 28
Station IDs have been renumbered to 0..27 to match list indices.


In [20]:
import gzip
import shutil
import os

# --- Configuration ---
input_filename = 'TD_W34_filtered_28_stations.json' 
output_filename = input_filename + '.gz'

# --- Processing ---
# Compress the file and save it in the current directory
try:
    with open(input_filename, 'rb') as f_in:
        with gzip.open(output_filename, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    
    # Get the absolute path to print it out for you
    full_path = os.path.abspath(output_filename)
    print(f"Success! File saved at:\n{full_path}")

except FileNotFoundError:
    print(f"Error: Could not find '{input_filename}'. Make sure it is in the same folder as this script.")

Success! File saved at:
/Users/ingvildvs/Documents/FOMOsim/policies/sjovik_sund/output/instance_data/TD_W34_filtered_28_stations.json.gz
